In [1]:
!git clone https://github.com/prava241/flash-attention.git
%cd flash-attention

Cloning into 'flash-attention'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 39 (delta 12), reused 34 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 9.50 KiB | 4.75 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/flash-attention


In [184]:
!git pull origin main
# !mkdir data
# !git checkout main

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 8 (delta 4), reused 8 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.75 KiB | 898.00 KiB/s, done.
From https://github.com/prava241/flash-attention
 * branch            main       -> FETCH_HEAD
   afb7205..0004451  main       -> origin/main
Updating afb7205..0004451
Fast-forward
 src/{main.cu => baseline.cu} | 191 ++++++++++---------------------------------
 src/kernels.cu               |   1 -
 src/tiled.cu                 | 127 ++++++++++++++++++++++++++++
 src/utils.cu                 |  42 ++++++++++
 src/utils.h                  |  17 ++++
 5 files changed, 229 insertions(+), 149 deletions(-)
 rename src/{main.cu => baseline.cu} (61%)
 create mode 100644 src/tiled.cu
 create mode 100644 src/utils.cu
 create mode 100644 src/utils.h


In [185]:
import torch
import numpy as np

N = 4096
D = 256

Q = torch.randn(N, D, device="cuda")
K = torch.randn(N, D, device="cuda")
V = torch.randn(N, D, device="cuda")

Q.cpu().numpy().astype(np.float32).tofile("data/q.bin")
K.cpu().numpy().astype(np.float32).tofile("data/k.bin")
V.cpu().numpy().astype(np.float32).tofile("data/v.bin")

In [19]:
!ls data

k.bin  output.bin  q.bin  v.bin


In [215]:
import math

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()

# Naive PyTorch
S = Q @ K.T
S = S / math.sqrt(D)
P = torch.softmax(S, dim=-1)
O = P @ V

#Optimized PyTorch
# O = torch.nn.functional.scaled_dot_product_attention(Q, K, V)

end.record()
torch.cuda.synchronize()
elapsed_ms = start.elapsed_time(end)

print(f"pytorch_attention runtime: {elapsed_ms:.3f} ms")

pytorch_attention runtime: 8.363 ms
Max abs error: 3.874302e-07
Mean abs error: 2.1585064e-08
All close: True


In [187]:
# Saves output to data/output.bin, reports runtime
# TODO: Profiles for bottlenecks
!nvcc -O3 src/baseline.cu src/kernels.cu src/utils.cu -o baseline_test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./baseline_test 4096 256

reference = O.cpu().numpy()
output = np.fromfile("data/output.bin", dtype=np.float32).reshape(N, D)

print("Max abs error:", np.max(np.abs(reference - output)))
print("Mean abs error:", np.mean(np.abs(reference - output)))
print("All close:", np.allclose(reference, output, atol=1e-5, rtol=1e-5))

baseline_attention runtime: 190.365 ms


In [197]:
!nvcc -O3 src/tiled.cu src/kernels.cu src/utils.cu -o tiled_test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./tiled_test 4096 256

reference = O.cpu().numpy()
output = np.fromfile("data/output.bin", dtype=np.float32).reshape(N, D)

print("Max abs error:", np.max(np.abs(reference - output)))
print("Mean abs error:", np.mean(np.abs(reference - output)))
print("All close:", np.allclose(reference, output, atol=1e-5, rtol=1e-5))

baseline_attention runtime: 115.789 ms
